# Win Determinants for NBA Teams

## Stat 306 Group P4 
#### **Group Members:** ####
- Russell Gibriel
- Sadila Liyanage Jayasinghe
- Chaitanya Thakral
- Geoff Zheng

## Background
Regular season wins measure team success, but teams achieve wins through a mix of offensive strategy (shooting, playmaking) and defensive actions (steals, blocks, rebounds). Using the historical NBA team-season data from the Kaggle “NBA Stats (1947–present)” dataset, we test which group of commonly reported per game metrics is more strongly associated with season wins after controlling for the other variables.

## Data Source and Collection
The data for this project was obtained from the Kaggle dataset “NBA Stats (1947-present)” by Sumitro Datta , using the files Team Summaries.csv and Team Stats per Game.csv. The dataset was compiled by scraping historical NBA team statistics from Basketball-Reference (Sports-Reference), which publishes team results and box score derived team metrics from each season. Some stats here are not tracked until later seasons. The two datasets will be merged since variables from both datasets will be used.

Datta , S. (2026, February 1). NBA Stats (1947-present). [Data set]. Kaggle. https://www.kaggle.com/datasets/sumitrodatta/nba-aba-baa-stats

## Variable Descriptions
| Role        | Variable Name | Variable Type       | Description                              |
|--------------|---------------|---------------------|------------------------------------------|
| Response     | w             | Numerical (integer) | Number of wins in that season            |
| Explanatory  | pts_per_game  | Numerical (double)  | Points per game scored by the team       |
| Explanatory  | ast_per_game  | Numerical (double)  | Assists per game by the team             |
| Explanatory  | fg_percent    | Numerical (double)  | Field goal percentage                    |
| Explanatory  | stl_per_game  | Numerical (double)  | Number of steals per game                |
| Explanatory  | blk_per_game  | Numerical (double)  | Blocks per game                          |
| Explanatory  | trb_per_game  | Numerical (double)  | Total rebounds per game                  |

## Research Question
Holding the other metrics constant, which group of per-game metrics has a stronger association with the seasonal wins (w): offensive performance (pts_per_game, ast_per_game, and fg_percent) or defensive performance (stl_per_game, blk_per_game, and trb_per_game).


In [1]:
# load necessary libraries for data analysis
library(tidyverse)
library(repr)
library(broom) # needed for tidy
library(car) # needed for vif

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: carData


Attaching package: ‘car’


The following object is masked from ‘package:dplyr’:

    recode


The following object is masked from ‘package:purrr’:

    some




In [3]:
# the data is stored in a personal github repository and loaded from there
url1 <- "https://raw.githubusercontent.com/Russell-97/NBA-Teams-Win-Determinants/refs/heads/main/Data/Team%20Stats%20Per%20Game.csv"
url2 <- "https://raw.githubusercontent.com/Russell-97/NBA-Teams-Win-Determinants/refs/heads/main/Data/Team%20Summaries.csv"
per_game_data <- read_csv(url1)
summary_data <- read_csv(url2)

Rows: 1907 Columns: 28
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (3): lg, team, abbreviation
dbl (24): season, g, mp_per_game, fg_per_game, fga_per_game, fg_percent, x3p...
lgl  (1): playoffs

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 1907 Columns: 31
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (4): lg, team, abbreviation, arena
dbl (26): season, age, w, l, pw, pl, mov, sos, srs, o_rtg, d_rtg, n_rtg, pac...
lgl  (1): playoffs

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [5]:
# Drop the League Average rows from both datasets
per_game_data  <- per_game_data  |> filter(team != "League Average")
summary_data <- summary_data |> filter(team != "League Average")

# Join wins (w) from summary dataset into per_game dataset
merged_data <- per_game_data |>
  left_join(
    summary_data |> select(season, abbreviation, w),
    by = c("season", "abbreviation")
  )

head(merged_data)

season,lg,team,abbreviation,playoffs,g,mp_per_game,fg_per_game,fga_per_game,fg_percent,⋯,orb_per_game,drb_per_game,trb_per_game,ast_per_game,stl_per_game,blk_per_game,tov_per_game,pf_per_game,pts_per_game,w
<dbl>,<chr>,<chr>,<chr>,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2026,NBA,Atlanta Hawks,ATL,FALSE,61,240.8,43.0,91.8,0.468,⋯,10.4,32.7,43.1,30.3,9.4,4.7,14.3,19.8,117.1,30
2026,NBA,Boston Celtics,BOS,FALSE,59,240.8,42.6,91.0,0.468,⋯,12.7,33.1,45.8,24.4,7.5,5.4,12.2,19.5,115.0,39
2026,NBA,Brooklyn Nets,BRK,FALSE,59,241.3,37.8,85.1,0.445,⋯,10.8,29.3,40.2,25.5,7.7,4.2,15.5,20.2,107.0,15
2026,NBA,Chicago Bulls,CHI,FALSE,60,240.8,42.1,89.8,0.469,⋯,10.2,34.3,44.5,28.8,7.5,5.0,15.1,18.7,115.7,24
2026,NBA,Charlotte Hornets,CHO,FALSE,61,241.2,40.9,88.8,0.461,⋯,12.8,33.5,46.3,26.6,7.0,4.5,15.9,19.0,116.0,30
2026,NBA,Cleveland Cavaliers,CLE,FALSE,61,241.2,43.4,91.1,0.476,⋯,12.1,32.4,44.6,28.5,9.0,5.1,14.4,19.8,119.4,37


In [12]:
data <- merged_data |>
  select(season, team, abbreviation, w, pts_per_game, ast_per_game, fg_percent, 
         stl_per_game, blk_per_game, trb_per_game) |>
  drop_na()   # remove rows with any missing value in these columns

head(data)

season,team,abbreviation,w,pts_per_game,ast_per_game,fg_percent,stl_per_game,blk_per_game,trb_per_game
<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2026,Atlanta Hawks,ATL,30,117.1,30.3,0.468,9.4,4.7,43.1
2026,Boston Celtics,BOS,39,115.0,24.4,0.468,7.5,5.4,45.8
2026,Brooklyn Nets,BRK,15,107.0,25.5,0.445,7.7,4.2,40.2
2026,Chicago Bulls,CHI,24,115.7,28.8,0.469,7.5,5.0,44.5
2026,Charlotte Hornets,CHO,30,116.0,26.6,0.461,7.0,4.5,46.3
2026,Cleveland Cavaliers,CLE,37,119.4,28.5,0.476,9.0,5.1,44.6


In [19]:
reg_offensive  <- lm(w ~ pts_per_game + ast_per_game + fg_percent, data = data)
reg_defensive  <- lm(w ~ stl_per_game + blk_per_game + trb_per_game, data = data)
reg_full <- lm(w ~ pts_per_game + ast_per_game + fg_percent +
                   stl_per_game + blk_per_game + trb_per_game, data = data)

print("Offensive model")
print(summary(reg_offensive))

print("Defensive model")
print(summary(reg_defensive))

print("Full model")
print(summary(reg_full))

[1] "Offensive model"

Call:
lm(formula = w ~ pts_per_game + ast_per_game + fg_percent, data = data)

Residuals:
    Min      1Q  Median      3Q     Max 
-39.887  -7.812   0.455   7.865  29.201 

Coefficients:
               Estimate Std. Error t value Pr(>|t|)    
(Intercept)  -109.49613    7.10293 -15.416   <2e-16 ***
pts_per_game   -0.10803    0.05886  -1.835   0.0667 .  
ast_per_game   -0.36836    0.17382  -2.119   0.0342 *  
fg_percent    365.80583   20.87241  17.526   <2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 10.84 on 1459 degrees of freedom
Multiple R-squared:  0.2543,	Adjusted R-squared:  0.2528 
F-statistic: 165.9 on 3 and 1459 DF,  p-value: < 2.2e-16

[1] "Defensive model"

Call:
lm(formula = w ~ stl_per_game + blk_per_game + trb_per_game, 
    data = data)

Residuals:
    Min      1Q  Median      3Q     Max 
-43.678  -9.182   0.662   8.402  31.992 

Coefficients:
             Estimate Std. Error t value Pr(>|t|)  